# Generate simulations from best parameters

This notebook uses samples of the posterior distribution inferred by a trained density estimator to generate the data structure and files necessary to launch `n_sim` simulations using the best-inferred parameters.

This has been used to simulate the populations to compare with observations in Figures 3, 4, 7 and 8 in Ronchi et al. (2026).

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/experiments_paper.zip`, copy it into the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026` and unpack the file so that the results data will be saved in the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper`.

In [ ]:
import torch
import corner
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import json

from mlpoppyns.learning.utils.import_param_statistics import import_statistics

In [ ]:
root_path = "../../data/paper_results/ronchi_etal_2026/experiments_paper"

# By default, we consider the inference results on the entire X-ray sample.
# To consider the inference results using only young magnetars and XDINSs
# and reproduce Figures 7 and 8 set `use_young_xdins_only` to True.

use_young_xdins_only = True

if use_young_xdins_only:
    training_exp_path = (
        f"{root_path}/tsnpe_experiment_1_maps8_res32_youngxdins"
    )
    posterior_samples_path = f"{training_exp_path}/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4"
else:
    training_exp_path = f"{root_path}/tsnpe_experiment_1_maps8_res32"
    posterior_samples_path = f"{training_exp_path}/learning/models/SBI_ConvolutionMDN/20260115_142235/round_4"

stats_path = f"{training_exp_path}/data/statistics_train.json"

In [ ]:
posterior_samples = (
    torch.load(f"{posterior_samples_path}/samples_posterior.pt")
    .detach()
    .cpu()
    .numpy()
)
print(np.shape(posterior_samples))
n_param = np.shape(posterior_samples)[1]

In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)
print(np.shape(mean))

In [ ]:
posterior_samples = posterior_samples * std[0:n_param] + mean[0:n_param]

Select `n_sim` random samples from the posterior.

In [ ]:
n_sim = 100
indices = np.random.choice(
    posterior_samples.shape[0], size=n_sim, replace=False
)
params_sim = posterior_samples[indices]
print(params_sim)
print(np.shape(params_sim))

In [ ]:
keys = [
    "P_initial_log10_mean",
    "P_initial_log10_sigma",
    "B_initial_log10_mean_comp1",
    "B_initial_log10_sigma_comp1",
    "B_initial_log10_mean_comp2",
    "B_initial_log10_sigma_comp2",
    "B_initial_log10_weight_comp1",
    "a_late",
    "L_radio_log10_mean",
    "epsilon_L",
]

In [ ]:
simulations_path = f"{training_exp_path}/simulations_best_params"
print(simulations_path)

In [ ]:
# Loop through the n_sim rows and create the folder structure to save the output of the simulations.
for i, values in enumerate(params_sim):

    # Create the dictionary for this row
    d = dict(zip(keys, values))

    # Folder name with zero-padding to 6 digits.
    folder_name = f"{simulations_path}/output_simulations/{i:06d}"  # produces 000000, 000001, ..., 000099

    # Create folder if it doesn't exist
    os.makedirs(folder_name, exist_ok=True)

    # Save override.json file with the paparmeters of each simulation inside the folder.
    file_path = os.path.join(folder_name, "override.json")
    with open(file_path, "w") as f:
        json.dump(d, f, indent=4)

In [ ]:
# Create an arguments.txt file containing the arguments to run each simulation.
arguments_path = f"{simulations_path}/htcondor_submit"

with open(f"{arguments_path}/arguments.txt", "w") as f:
    for i in range(n_sim):  # 000000 → 000099
        folder_path = f"{simulations_path}/output_simulations/{i:06d}"
        override_path = f"{folder_path}/override.json"
        f.write(f"{folder_path} {override_path}\n")